# 00 — Environment and input readiness

**Question:** Is this runtime capable of reproducing the documented NewAthena X-IFU extended-source simulation?

A passing result establishes execution readiness for the frozen baseline. It does not by itself establish scientific readiness.

In [ ]:
import json
import subprocess
import sys

from newathena_sixte_extended_sources import load_workspace

WORKSPACE = load_workspace()
ROOT = WORKSPACE.root
print({'project_root': str(ROOT), 'runtime': str(WORKSPACE.runtime),
       'inputs': str(WORKSPACE.inputs), 'profile': WORKSPACE.profile,
       'science_status': WORKSPACE.profile_values['science_status']})

## Runtime and archive gate

The checker verifies frozen archive checksums, installed executables, workshop inputs, and the official X-IFU bundle.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(ROOT / 'scripts/check_phase1_readiness.py')],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
report = json.loads(completed.stdout)
assert report['core_runtime_ready']
assert report['xifu_instrument_bundle']['status'] == 'verified'
assert all(item['ok'] for item in report['archive_checksums'].values())
assert all(report['required_paths'].values())
print('READINESS: PASS', report['versions'])

## Baseline science configuration

The baseline uses X-IFU 1.11.1, the ESA 4 eV response, a deterministic seed, and a spectrum padded beyond the ARF limits.

In [ ]:
import yaml
configuration = yaml.safe_load((ROOT / 'config/phase2-baseline.yml').read_text())
assert configuration['instrument']['version'] == '1.11.1'
assert configuration['instrument']['energy_resolution'].startswith('4 eV')
assert configuration['spectral_grid']['minimum_keV'] < 0.15
assert configuration['spectral_grid']['maximum_keV'] > 12.100345
assert configuration['simulation']['seed'] == 20260721
configuration

## Try it: portability check

**Exercise.** Set `NEWATHENA_PROFILE=teaching`, restart the kernel, and rerun the first cell. Which compute settings change, and which instrument/science assumptions remain fixed?

<details><summary>Solution and interpretation</summary>

Only exposure and ARF Monte Carlo budgets change. Instrument release, source truth, fitting statistic, and validation definitions do not. A teaching profile is a workflow demonstration, not a replacement reference result. If discovery fails, set `NEWATHENA_PROJECT_ROOT`; hosted sessions should also set `NEWATHENA_RUNTIME_DIR` and `NEWATHENA_INPUT_DIR`.
</details>

## Interpretation

The remaining known metadata warning is that upstream response files lack a `FILTER` keyword; the no-filter products record `FILTER=NONE`.